# **BirdCLEF 2025 Data Preprocessing Notebook**
This notebook demonstrates how we can transform audio data into mel-spectrogram data. This transformation is essential for training 2D Convolutional Neural Networks (CNNs) on audio data, as it converts the one-dimensional audio signals into two-dimensional image-like representations.
I run this public notebook in debug mode(only a few sample processing). You can find the fully preprocessed mel spectrogram training dataset here --> [BirdCLEF'25 | Mel Spectrograms](https://www.kaggle.com/datasets/kadircandrisolu/birdclef25-mel-spectrograms).


In [ ]:
# %%capture
# !pip install biodenoising --quiet

In [2]:
import os
import csv
import cv2
import math
import time
import librosa
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
import hashlib

import torch
import warnings
warnings.filterwarnings("ignore")

In [3]:
import tensorflow_hub as hub
import tensorflow as tf
tf.experimental.numpy.experimental_enable_numpy_behavior()

2025-04-28 09:15:40.556491: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-28 09:15:40.565214: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1745799340.574500    2995 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1745799340.577518    2995 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1745799340.585775    2995 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [ ]:
# import torchaudio
# from torchaudio.transforms import MelSpectrogram, AmplitudeToDB

# from biodenoising import pretrained
# from biodenoising.denoiser.dsp import convert_audio

In [5]:
class Config:
 
    DEBUG_MODE = True
    
    OUTPUT_DIR = ''
    DATA_ROOT = 'dataset/'
    FS = 16000 # used to be 32000
    
    # Mel spectrogram parameters
    N_FFT = 1024
    HOP_LENGTH = 512
    N_MELS = 128
    FMIN = 40
    FMAX = 15000
    
    TARGET_DURATION = 5.0
    TARGET_SHAPE = (256, 256)  
    
    N_MAX = 50 if DEBUG_MODE else None

    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

config = Config()

In [6]:
print(f"Debug mode: {'ON' if config.DEBUG_MODE else 'OFF'}")
print(f"Max samples to process: {config.N_MAX if config.N_MAX is not None else 'ALL'}")

print("Loading taxonomy data...")
taxonomy_df = pd.read_csv(f'{config.DATA_ROOT}/taxonomy.csv')
species_class_map = dict(zip(taxonomy_df['primary_label'], taxonomy_df['class_name']))

print("Loading training metadata...")
train_df = pd.read_csv(f'{config.DATA_ROOT}/train.csv')

Debug mode: ON
Max samples to process: 50
Loading taxonomy data...
Loading training metadata...


In [ ]:
label_list = sorted(train_df['primary_label'].unique())
label_id_list = list(range(len(label_list)))
label2id = dict(zip(label_list, label_id_list))
id2label = dict(zip(label_id_list, label_list))

print(f'Found {len(label_list)} unique species')
working_df = train_df[['primary_label', 'secondary_labels', 'rating', 'filename']].copy()
working_df['target'] = working_df.primary_label.map(label2id)
working_df['filepath'] = config.DATA_ROOT + '/train_audio/' + working_df.filename
working_df['samplename'] = working_df.filename.map(lambda x: x.split('/')[0] + '-' + x.split('/')[-1].split('.')[0])
working_df['class'] = working_df.primary_label.map(lambda x: species_class_map.get(x, 'Unknown'))
total_samples = min(len(working_df), config.N_MAX or len(working_df))
print(f'Total samples to process: {total_samples} out of {len(working_df)} available')
print(f'Samples by class:')
print(working_df['class'].value_counts())

Found 206 unique species
Total samples to process: 50 out of 28564 available
Samples by class:
class
Aves        27648
Amphibia      583
Mammalia      178
Insecta       155
Name: count, dtype: int64


In [9]:
def audio2melspec(audio_data):
    if np.isnan(audio_data).any():
        mean_signal = np.nanmean(audio_data)
        audio_data = np.nan_to_num(audio_data, nan=mean_signal)

    mel_spec = librosa.feature.melspectrogram(
        y=audio_data,
        sr=config.FS,
        n_fft=config.N_FFT,
        hop_length=config.HOP_LENGTH,
        n_mels=config.N_MELS,
        fmin=config.FMIN,
        fmax=config.FMAX,
        power=2.0
    )

    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
    mel_spec_norm = (mel_spec_db - mel_spec_db.min()) / (mel_spec_db.max() - mel_spec_db.min() + 1e-8)
    
    return mel_spec_norm

In [8]:
working_df['secondary_labels'] = working_df['secondary_labels'].apply(lambda x: x[1:-1].split(','))

In [9]:
# 1. Remove entries with duplicate audio files

def find_duplicate_audio_files(df):
    """Identify duplicate audio files using MD5 hashing"""
    print("\nChecking for duplicate audio files...")
    
    file_hashes = {}
    duplicates = set()
    
    for idx, row in tqdm(df.iterrows(), total=len(df)):
        filepath = row['filepath']
        try:
            with open(filepath, 'rb') as f:
                file_hash = hashlib.md5(f.read()).hexdigest()
            
            if file_hash in file_hashes:
                duplicates.add(idx)
                
                original_idx = file_hashes[file_hash]
                if original_idx not in duplicates:
                    duplicates.add(original_idx)
            else:
                file_hashes[file_hash] = idx
        except Exception as e:
            print(f"Error processing {filepath}: {e}")
    
    return duplicates

original_count = len(working_df)
duplicate_indices = find_duplicate_audio_files(working_df)
working_df = working_df.drop(index=duplicate_indices).reset_index(drop=True)
print(f"Removed {len(duplicate_indices)} duplicate files ({original_count - len(working_df)} rows)")


Checking for duplicate audio files...


  0%|          | 0/28564 [00:00<?, ?it/s]

Removed 0 duplicate files (0 rows)


In [10]:
perch_model = hub.load('https://www.kaggle.com/models/google/bird-vocalization-classifier/TensorFlow2/bird-vocalization-classifier/8')

I0000 00:00:1745799391.393477    2995 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5520 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4070 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


In [11]:
labels_path = hub.resolve('https://www.kaggle.com/models/google/bird-vocalization-classifier/TensorFlow2/bird-vocalization-classifier/8') + "/assets/label.csv"
perch_labels = pd.read_csv(labels_path)
perch_labels = perch_labels.rename(columns={ perch_labels.columns[0]: 'name' })

In [13]:
def check_label_with_perch(audio_data, model, primary_label, secondary_labels, target_length=160000):
    """Process audio in 10-second chunks compatible with Perch"""
    try:
        if not hasattr(model, 'infer_tf'):
            raise ValueError("Model object doesn't have infer_tf method")
        
        if len(audio_data) < target_length:
            padded_audio = np.pad(audio_data, (0, target_length - len(audio_data)))
            chunks = [padded_audio]
        else:
            num_chunks = int(np.ceil(len(audio_data) / target_length))
            chunks = np.array_split(audio_data[:num_chunks * target_length], num_chunks)
        
        all_outputs = []
        for chunk in chunks:
            chunk = chunk[:target_length]
            if len(chunk) < target_length:
                chunk = np.pad(chunk, (0, target_length - len(chunk)))
            
            outputs = model.infer_tf(chunk[np.newaxis, :])
            all_outputs.append(outputs)
        
        # Aggregate predictions across chunks
        avg_scores = np.mean([o['label'][0] for o in all_outputs], axis=0)
        
        # Get top prediction using external labels
        top_class_idx = np.argmax(avg_scores)
        try:
            top_class = perch_labels.iloc[top_class_idx]['name']
        except IndexError:
            print(f"Warning: Prediction index {top_class_idx} out of bounds for label file")
            top_class = f"unknown_class_{top_class_idx}"
        
        top_score = avg_scores[top_class_idx]
        
        # print(f"\nGround Truth:")
        # print(f"- Primary: {primary_label}")
        # print(f"- Secondary: {secondary_labels}")
        # print(f"\nPerch Prediction:")
        # print(f"- Top: {top_class} (Score: {top_score:.2f})")
        
        return {
            'matches_primary': top_class == primary_label,
            'matches_secondary': top_class in secondary_labels if secondary_labels else False,
            'top_class': top_class,
            'top_score': top_score,
            'label_scores': avg_scores
        }
        
    except Exception as e:
        print(f"Perch processing failed: {e}")
        return None

In [14]:
# 2. Separate to 10 second chunks then use cyclic padding if the last chunk isn't 10 seconds long

def process_audio_to_10s_chunks(filepath, target_duration=10.0, sr=32000):
    """Load audio and split into exact 10-second chunks with CYCLIC PADDING if needed"""
    audio_data, _ = librosa.load(filepath, sr=sr)
    target_samples = int(target_duration * sr)
    
    # For files shorter than target duration
    if len(audio_data) < target_samples:
        # Cyclic padding by repeating the audio
        repeats = int(np.ceil(target_samples / len(audio_data)))
        padded_audio = np.tile(audio_data, repeats)[:target_samples]
        return [padded_audio]
    
    # For longer files
    num_chunks = int(np.ceil(len(audio_data) / target_samples))
    chunks = []
    for i in range(num_chunks):
        start = i * target_samples
        end = start + target_samples
        chunk = audio_data[start:end]
        
        # Cyclic padding for the last chunk if needed
        if len(chunk) < target_samples:
            remaining = target_samples - len(chunk)
            chunk = np.concatenate([
                chunk,
                audio_data[:remaining]  # Wrap around to beginning
            ])
        
        chunks.append(chunk)
    
    return chunks

# Create new expanded DataFrame
expanded_rows = []
for working_df_idx, row in tqdm(working_df.iterrows(), total=len(working_df)):
    chunks = process_audio_to_10s_chunks(row.filepath, target_duration=10.0, sr=config.FS)
    for i, chunk in enumerate(chunks):

        label_info = check_label_with_perch(chunk, perch_model, row.primary_label, row.secondary_labels)
        
        expanded_row = {
            'working_df_idx': working_df_idx,
            'original_samplename': row.samplename,
            'chunk_id': i,
            'filepath': row.filepath,
            'primary_label': row.primary_label,
            'secondary_labels': row.secondary_labels,
            'matches_primary': label_info['matches_primary'], 
            'matches_secondary': label_info['matches_primary'], 
            'top_class': label_info['top_class'],
        }

expanded_df = pd.DataFrame(expanded_rows)
print(f"Original {len(working_df)} files → {len(expanded_df)} 10-second chunks")

  0%|          | 0/28564 [00:00<?, ?it/s]

I0000 00:00:1745799518.602151    2995 service.cc:152] XLA service 0x3b962cb0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1745799518.602201    2995 service.cc:160]   StreamExecutor device (0): NVIDIA GeForce RTX 4070 Laptop GPU, Compute Capability 8.9
2025-04-28 09:18:38.757238: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-04-28 09:18:38.764840: W tensorflow/compiler/tf2xla/kernels/assert_op.cc:39] Ignoring Assert operator jax2tf_infer_fn_/assert_equal_1/Assert/AssertGuard/Assert
E0000 00:00:1745799519.229498    2995 cuda_dnn.cc:522] Loaded runtime CuDNN library: 9.1.0 but source was compiled with: 9.3.0.  CuDNN library needs to have matching major version and equal or higher minor version. If using a binary install, upgrade your CuDNN library.  If building from sources, make sure the library loaded at runtime is com

Perch processing failed: DNN library initialization failed. Look at the errors above for more details. [Op:__inference_restored_function_body_20537]


TypeError: 'NoneType' object is not subscriptable

In [ ]:
print("Starting audio processing...")
print(f"{'DEBUG MODE - Processing only 50 samples' if config.DEBUG_MODE else 'FULL MODE - Processing all samples'}")
start_time = time.time()

all_bird_data = {}
errors = []

for i, row in tqdm(working_df.iterrows(), total=total_samples):
    if config.N_MAX is not None and i >= config.N_MAX:
        break
    
    try:
        audio_data, _ = librosa.load(row.filepath, sr=config.FS)
        primary_label = row.primary_label
        
        secondary_labels = [e.replace("'", '') for e in row.secondary_labels]
        secondary_labels = None if secondary_labels[0] == '' else secondary_labels

        print(check_label_with_perch(audio_data, perch_model, primary_label, secondary_labels)['matches_primary'])

        continue

        target_samples = int(config.TARGET_DURATION * config.FS)

        if len(audio_data) < target_samples:
            n_copy = math.ceil(target_samples / len(audio_data))
            if n_copy > 1:
                audio_data = np.concatenate([audio_data] * n_copy)

        start_idx = max(0, int(len(audio_data) / 2 - target_samples / 2))
        end_idx = min(len(audio_data), start_idx + target_samples)
        center_audio = audio_data[start_idx:end_idx]

        if len(center_audio) < target_samples:
            center_audio = np.pad(center_audio, 
                                 (0, target_samples - len(center_audio)), 
                                 mode='constant')

        mel_spec = audio2melspec(center_audio)

        if mel_spec.shape != config.TARGET_SHAPE:
            mel_spec = cv2.resize(mel_spec, config.TARGET_SHAPE, interpolation=cv2.INTER_LINEAR)

        all_bird_data[row.samplename] = mel_spec.astype(np.float32)
        
    except Exception as e:
        print(f"Error processing {row.filepath}: {e}")
        errors.append((row.filepath, str(e)))

end_time = time.time()
print(f"Processing completed in {end_time - start_time:.2f} seconds")
print(f"Successfully processed {len(all_bird_data)} files out of {total_samples} total")
print(f"Failed to process {len(errors)} files")

In [ ]:
import matplotlib.pyplot as plt

samples = []
displayed_classes = set()

max_samples = min(4, len(all_bird_data))

for i, row in working_df.iterrows():
    if i >= (config.N_MAX or len(working_df)):
        break
        
    if row['samplename'] in all_bird_data:
        if config.DEBUG_MODE:
            if row['class'] not in displayed_classes:
                samples.append((row['samplename'], row['class'], row['primary_label']))
                displayed_classes.add(row['class'])
        else:
            if row['class'] not in displayed_classes:
                samples.append((row['samplename'], row['class'], row['primary_label']))
                displayed_classes.add(row['class'])
        
        if len(samples) >= max_samples:  
            break

if samples:
    plt.figure(figsize=(16, 12))
    
    for i, (samplename, class_name, species) in enumerate(samples):
        plt.subplot(2, 2, i+1)
        plt.imshow(all_bird_data[samplename], aspect='auto', origin='lower', cmap='viridis')
        plt.title(f"{class_name}: {species}")
        plt.colorbar(format='%+2.0f dB')
    
    plt.tight_layout()
    debug_note = "debug_" if config.DEBUG_MODE else ""
    plt.savefig(f'{debug_note}melspec_examples_baseline.png')
    plt.show()